# Copy-Trading Backtest Analysis

For a given wallet, this notebook:
1. Loads the whale's trade history and enriches it with P&L from resolved markets
2. Simulates copying each trade at configurable time offsets using real order book snapshots
3. Computes our P&L for each copy attempt (same sizing model: proportional to their wallet fraction)
4. Shows a side-by-side trade-level comparison and summary metrics

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../../..").resolve() / ".env")

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
print("Setup complete.")

## Configuration

Set the wallet (username or proxy address), capital parameters, and backtest window here.

In [ ]:
# ── Wallet ────────────────────────────────────────────────────────────────────
WALLET = "xdd07070"          # Username or 0x proxy wallet address

# ── Capital sizing ────────────────────────────────────────────────────────────
WHALE_CAPITAL   = 10_000     # Estimated total USD wallet of the whale
OUR_CAPITAL     = 500        # USD we have allocated to copy-trade this wallet

# ── Time window ───────────────────────────────────────────────────────────────
DAYS = 30                    # How many days of history to load

# ── Copy offsets to simulate ──────────────────────────────────────────────────
OFFSETS_MS      = [500, 1_000, 5_000]   # ms after whale trade to attempt copy
PRIMARY_OFFSET  = 1_000                  # Which offset to use for the main table

# ── Databases ─────────────────────────────────────────────────────────────────
WHALE_DB_URL = os.environ.get("WHALE_DB_URL", "")
TICK_DB_URL  = os.environ.get("TICK_DB_URL",  "sqlite+aiosqlite:///tick_data.db")

# ── Order book snapshot search tolerance ──────────────────────────────────────
OB_TOLERANCE_MS = 2_000

if not WHALE_DB_URL:
    raise RuntimeError("WHALE_DB_URL is not set. Add it to your .env file.")

print(f"Wallet          : {WALLET}")
print(f"Capital sizing  : whale=${WHALE_CAPITAL:,}  ours=${OUR_CAPITAL:,}")
print(f"History         : {DAYS} days")
print(f"Offsets (ms)    : {OFFSETS_MS}")

## Step 1 — Resolve wallet address

If `WALLET` is a username, look up the proxy wallet address via the Polymarket leaderboard API.

In [ ]:
from trading_tools.clients.polymarket.client import PolymarketClient

if WALLET.startswith("0x"):
    address = WALLET.lower()
    username = address[:10] + "..."
else:
    async with PolymarketClient() as client:
        address = await client.lookup_user_address(WALLET)
    if not address:
        raise RuntimeError(f"Could not resolve username: {WALLET!r}")
    username = WALLET

print(f"Username : {username}")
print(f"Address  : {address}")

## Step 2 — Load whale trades and enrich with P&L

Fetches raw trades from the whale DB, then enriches each one with Gamma market metadata so we know the winning outcome and per-trade P&L for resolved markets.

In [ ]:
from decimal import Decimal

from trading_tools.apps.whale_monitor.enricher import MarketMetadata, enrich_trades
from trading_tools.apps.whale_monitor.repository import WhaleRepository

_SECONDS_PER_DAY = 86_400

now_ts  = int(time.time())
start_ts = now_ts - DAYS * _SECONDS_PER_DAY

whale_repo = WhaleRepository(WHALE_DB_URL)
await whale_repo.init_db()
raw_trades = await whale_repo.get_trades(address, start_ts, now_ts)
await whale_repo.close()

enrich_cache: dict[str, MarketMetadata] = {}
async with PolymarketClient() as client:
    enriched_trades = await enrich_trades(client, raw_trades, cache=enrich_cache)

resolved  = [t for t in enriched_trades if not t.is_active]
active    = [t for t in enriched_trades if t.is_active]

print(f"Total trades : {len(enriched_trades)}")
print(f"  Resolved   : {len(resolved)}")
print(f"  Active     : {len(active)}")

## Step 3 — Run copy backtest

For each whale trade, query the nearest order book snapshot at each offset and simulate buying with our proportional budget.

In [ ]:
from trading_tools.apps.tick_collector.repository import TickRepository
from trading_tools.apps.whale_copy_trader.copy_backtest_engine import CopyBacktestEngine

whale_repo2 = WhaleRepository(WHALE_DB_URL)
tick_repo   = TickRepository(TICK_DB_URL)
await whale_repo2.init_db()

engine = CopyBacktestEngine(
    whale_repo=whale_repo2,
    tick_repo=tick_repo,
    ob_tolerance_ms=OB_TOLERANCE_MS,
)

backtest_result = await engine.run(
    whale_address=address,
    start_ts=start_ts,
    end_ts=now_ts,
    whale_capital=Decimal(WHALE_CAPITAL),
    our_capital=Decimal(OUR_CAPITAL),
    offsets_ms=OFFSETS_MS,
)

await whale_repo2.close()
await tick_repo.close()

print(f"Backtest complete: {backtest_result.total_trades} trades simulated")
for offset_ms, stats in backtest_result.stats_by_offset.items():
    print(
        f"  +{offset_ms:>5}ms  OB found={stats.trades_with_book}/{stats.trades_total}"
        f"  full deploy={stats.trades_fully_deployed}"
        f"  avg slip={float(stats.avg_slippage):.4f}"
        f"  ({float(stats.avg_slippage_bps):.1f}bp)"
    )

## Step 4 — Build side-by-side comparison table

Join enriched whale trades with copy backtest results by (condition_id, timestamp).
Compute our P&L using the same formula as the whale:
`direction * fill_qty * (resolution_value - fill_price)`

In [ ]:
from datetime import UTC, datetime

# Index enriched trades by (condition_id, timestamp) for fast lookup
enriched_index = {
    (t.trade.condition_id, t.trade.timestamp): t
    for t in enriched_trades
}


def compute_our_pnl(attempt, side: str, outcome: str, winning_outcome: str | None) -> float | None:
    """Compute our P&L for a copy attempt given the market resolution."""
    if winning_outcome is None or float(attempt.fill_qty) == 0:
        return None
    direction = 1.0 if side.upper() == "BUY" else -1.0
    resolution_value = 1.0 if outcome.strip().lower() == winning_outcome.strip().lower() else 0.0
    return direction * float(attempt.fill_qty) * (resolution_value - float(attempt.fill_price))


rows = []
for bt in backtest_result.trades:
    enriched = enriched_index.get((bt.condition_id, bt.whale_ts))
    winning_outcome = enriched.winning_outcome if enriched else None
    is_active = enriched.is_active if enriched else True
    whale_pnl = enriched.trade_pnl if enriched else None

    row: dict = {
        "timestamp": datetime.fromtimestamp(bt.whale_ts, tz=UTC).strftime("%Y-%m-%d %H:%M"),
        "market": bt.title[:45],
        "outcome": bt.outcome,
        "side": bt.side,
        # ── Whale ─────────────────────────────────────────────────────────────
        "whale_size": round(float(bt.whale_size), 2),
        "whale_price": round(float(bt.whale_price), 4),
        "whale_cost_usd": round(float(bt.whale_trade_cost), 4),
        "whale_wallet_pct": round(float(bt.whale_fraction) * 100, 3),
        "whale_pnl": round(whale_pnl, 4) if whale_pnl is not None else None,
        "status": "active" if is_active else ("win" if (whale_pnl or 0) > 0 else "loss"),
    }

    # ── Our copy per offset ───────────────────────────────────────────────────
    for attempt in bt.attempts:
        pfx = f"our_{attempt.offset_ms}ms"
        our_pnl = compute_our_pnl(attempt, bt.side, bt.outcome, winning_outcome)
        row[f"{pfx}_budget_usd"]  = round(float(attempt.our_budget_usd), 4)
        row[f"{pfx}_fill_qty"]    = round(float(attempt.fill_qty), 4)
        row[f"{pfx}_fill_price"]  = round(float(attempt.fill_price), 4) if attempt.fill_qty > 0 else None
        row[f"{pfx}_cost_usd"]    = round(float(attempt.cost_usd), 4)
        row[f"{pfx}_slippage"]    = round(float(attempt.slippage), 4) if attempt.fill_qty > 0 else None
        row[f"{pfx}_pnl"]         = round(our_pnl, 4) if our_pnl is not None else None
        row[f"{pfx}_ob_found"]    = attempt.ob_found

    rows.append(row)

df = pd.DataFrame(rows)
print(f"Comparison table: {len(df)} rows, {len(df.columns)} columns")
df.head(3)

## Step 5 — Trade-level side-by-side view

Resolved trades only, at the primary offset. Shows whale vs our execution for each trade.

In [ ]:
pfx = f"our_{PRIMARY_OFFSET}ms"

view_cols = [
    "timestamp", "market", "outcome", "side", "status",
    # Whale
    "whale_size", "whale_price", "whale_cost_usd", "whale_wallet_pct", "whale_pnl",
    # Ours
    f"{pfx}_budget_usd", f"{pfx}_fill_qty", f"{pfx}_fill_price",
    f"{pfx}_cost_usd", f"{pfx}_slippage", f"{pfx}_pnl",
]

resolved_df = df[df["status"] != "active"][view_cols].copy()

resolved_df.columns = [
    "Time", "Market", "Outcome", "Side", "Result",
    "Whale Qty", "Whale Price", "Whale $", "Whale Wallet%", "Whale PnL",
    "Our Budget $", "Our Qty", "Our Price", "Our Spent $", "Slippage", "Our PnL",
]

# Colour wins green, losses red
def colour_result(val):
    if val == "win":  return "background-color: #d4edda; color: #155724"
    if val == "loss": return "background-color: #f8d7da; color: #721c24"
    return ""

def colour_pnl(val):
    if pd.isna(val): return ""
    return "color: #155724" if val > 0 else "color: #721c24"

display(
    resolved_df.style
        .applymap(colour_result, subset=["Result"])
        .applymap(colour_pnl, subset=["Whale PnL", "Our PnL"])
        .format(precision=4, na_rep="-")
        .set_caption(f"Resolved trades — copy at +{PRIMARY_OFFSET}ms (primary offset)")
)

## Step 6 — Win rate and PnL comparison

In [ ]:
import numpy as np

resolved = df[df["status"] != "active"].copy()
n_resolved = len(resolved)

# ── Whale stats ───────────────────────────────────────────────────────────────
whale_wins    = (resolved["whale_pnl"] > 0).sum()
whale_losses  = (resolved["whale_pnl"] < 0).sum()
whale_win_pct = whale_wins / n_resolved * 100 if n_resolved else 0
whale_total_pnl = resolved["whale_pnl"].sum()
whale_avg_pnl   = resolved["whale_pnl"].mean()

print("━" * 55)
print(f"{'':30s}  {'WHALE':>10}")
print("━" * 55)
print(f"{'Resolved trades':30s}  {n_resolved:>10}")
print(f"{'Wins':30s}  {whale_wins:>10}")
print(f"{'Losses':30s}  {whale_losses:>10}")
print(f"{'Win rate':30s}  {whale_win_pct:>9.1f}%")
print(f"{'Total PnL ($)':30s}  {whale_total_pnl:>10.2f}")
print(f"{'Avg PnL per trade ($)':30s}  {whale_avg_pnl:>10.4f}")
print("━" * 55)

# ── Our stats per offset ─────────────────────────────────────────────────────
print()
header = f"{'Metric':30s}" + "".join(f"  {ms:>8}ms" for ms in OFFSETS_MS)
print("━" * len(header))
print(header)
print("━" * len(header))

metrics = {}
for ms in OFFSETS_MS:
    col = f"our_{ms}ms_pnl"
    filled = resolved[col].notna()
    pnl_series = resolved.loc[filled, col]
    wins   = (pnl_series > 0).sum()
    losses = (pnl_series < 0).sum()
    n      = len(pnl_series)
    metrics[ms] = {
        "trades_with_fill": n,
        "wins":             wins,
        "losses":           losses,
        "win_pct":          wins / n * 100 if n else 0,
        "total_pnl":        pnl_series.sum() if n else 0,
        "avg_pnl":          pnl_series.mean() if n else np.nan,
    }

def _row(label, key, fmt):
    vals = "".join(f"  {fmt.format(metrics[ms][key]):>9}" for ms in OFFSETS_MS)
    print(f"{label:30s}{vals}")

_row("Trades with fill",    "trades_with_fill", "{:>9}")
_row("Wins",               "wins",             "{:>9}")
_row("Losses",             "losses",           "{:>9}")
_row("Win rate",           "win_pct",          "{:>8.1f}%")
_row("Total PnL ($)",      "total_pnl",        "{:>9.2f}")
_row("Avg PnL per trade",  "avg_pnl",          "{:>9.4f}")
print("━" * len(header))

## Step 7 — Offset comparison: fill rate and slippage

In [ ]:
offset_rows = []
for ms in OFFSETS_MS:
    stats = backtest_result.stats_by_offset[ms]
    m = metrics[ms]
    offset_rows.append({
        "Offset (ms)": ms,
        "OB found": f"{stats.trades_with_book}/{stats.trades_total}",
        "Full deploy": stats.trades_fully_deployed,
        "Partial": stats.trades_partial,
        "Avg deploy %": f"{float(stats.avg_budget_utilisation)*100:.1f}%",
        "Total deployed $": f"{float(stats.total_deployed_usd):.2f}",
        "Avg slippage": f"{float(stats.avg_slippage):.4f}",
        "Avg slip (bp)": f"{float(stats.avg_slippage_bps):.1f}",
        "Max slippage": f"{float(stats.max_slippage):.4f}",
        "Win rate": f"{m['win_pct']:.1f}%",
        "Total PnL $": f"{m['total_pnl']:.2f}",
    })

offset_df = pd.DataFrame(offset_rows).set_index("Offset (ms)")
display(offset_df)

## Step 8 — Cumulative PnL over time

In [ ]:
import matplotlib.pyplot as plt

plot_df = resolved.sort_values("timestamp").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))

# Whale cumulative PnL
ax.plot(
    plot_df.index,
    plot_df["whale_pnl"].cumsum(),
    label=f"Whale ({username})",
    linewidth=2,
    color="steelblue",
)

# Our cumulative PnL per offset
colours = ["tomato", "darkorange", "mediumseagreen"]
for ms, colour in zip(OFFSETS_MS, colours):
    col = f"our_{ms}ms_pnl"
    ax.plot(
        plot_df.index,
        plot_df[col].fillna(0).cumsum(),
        label=f"Us +{ms}ms",
        linestyle="--",
        linewidth=1.5,
        color=colour,
    )

ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_xlabel("Trade # (chronological)")
ax.set_ylabel("Cumulative PnL (USD)")
ax.set_title(f"Cumulative PnL: {username} vs copy strategies (${OUR_CAPITAL} allocated)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9 — Per-trade PnL scatter (whale vs us)

In [ ]:
pfx = f"our_{PRIMARY_OFFSET}ms"

# Only trades where we had a fill
both = resolved[resolved[f"{pfx}_pnl"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter whale pnl vs our pnl
ax = axes[0]
colours_scatter = both["status"].map({"win": "green", "loss": "red"})
ax.scatter(both["whale_pnl"], both[f"{pfx}_pnl"], c=colours_scatter, alpha=0.7, s=30)
ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
lim = max(abs(both["whale_pnl"]).max(), abs(both[f"{pfx}_pnl"]).max()) * 1.1
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("Whale PnL ($)")
ax.set_ylabel(f"Our PnL +{PRIMARY_OFFSET}ms ($)")
ax.set_title("Per-trade PnL: whale vs copy")
ax.grid(alpha=0.3)

# Right: PnL distribution
ax2 = axes[1]
ax2.hist(both["whale_pnl"].dropna(), bins=30, alpha=0.6, label="Whale", color="steelblue")
ax2.hist(both[f"{pfx}_pnl"].dropna(), bins=30, alpha=0.6, label=f"Us +{PRIMARY_OFFSET}ms", color="tomato")
ax2.axvline(0, color="grey", linewidth=0.8, linestyle=":")
ax2.set_xlabel("PnL ($)")
ax2.set_ylabel("Trade count")
ax2.set_title("PnL distribution")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10 — Slippage vs PnL: does slippage predict outcome?

In [ ]:
fig, axes = plt.subplots(1, len(OFFSETS_MS), figsize=(5 * len(OFFSETS_MS), 4), sharey=False)
if len(OFFSETS_MS) == 1:
    axes = [axes]

for ax, ms in zip(axes, OFFSETS_MS):
    pfx = f"our_{ms}ms"
    mask = resolved[f"{pfx}_pnl"].notna() & resolved[f"{pfx}_slippage"].notna()
    sub = resolved[mask]
    c = sub[f"{pfx}_pnl"].apply(lambda x: "green" if x > 0 else "red")
    ax.scatter(sub[f"{pfx}_slippage"], sub[f"{pfx}_pnl"], c=c, alpha=0.6, s=25)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Slippage (our price - whale price)")
    ax.set_ylabel("Our PnL ($)")
    ax.set_title(f"+{ms}ms")
    ax.grid(alpha=0.3)

fig.suptitle("Slippage vs PnL per offset", fontsize=13)
plt.tight_layout()
plt.show()

## Step 11 — Export full comparison table to CSV

In [ ]:
out_path = Path(f"copy_backtest_{username}_{DAYS}d.csv")
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path.resolve()}")